# TokCom TCLA vs H.265 — Full GPU Simulation
## Real TiTok Tokenizer + Prefix Reconstruction over Simulated WiFi Channel

**What this shows:**
- A real video (Big Buck Bunny) over a simulated WiFi interference event
- **H.265**: fixed bitrate encode → frame dropped when channel too weak → **❄ FREEZE**
- **TCLA**: tokenize each frame → send only `k` tokens → decoder fills the rest → **always moving**

**Pipeline:**
```
Channel:  SNR 30 dB ────── drops to 2 dB (2.5s–6s) ────── recovers 30 dB

H.265:    encode @ fixed bitrate ──► fits in channel? ──► OK / ❄ FREEZE
TCLA:     tokenize ──► k = f(channel) tokens sent ──► decode k + fill tail ──► always a frame
```

**Runtime on T4 GPU:** ~10–15 min for a 10 s video

---
*Switch to GPU: Runtime → Change runtime type → T4 GPU*

## 1 — GPU check

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected — go to Runtime > Change runtime type > T4 GPU')


## 2 — Install packages & clone repo

In [ ]:
import os, subprocess

# Python packages
os.system('pip install -q timm einops omegaconf huggingface_hub accelerate '
          'open_clip_torch imageio[ffmpeg] av safetensors')
print('packages: ok')

# Clone bytedance/1d-tokenizer — contains the TiTok model code
if not os.path.exists('1d-tokenizer'):
    os.system('git clone -q https://github.com/bytedance/1d-tokenizer.git')
ok = os.path.exists('1d-tokenizer/modeling/titok.py')
print(f'1d-tokenizer repo: {"ok" if ok else "MISSING — rerun this cell"}')
if not ok:
    raise RuntimeError('Repo clone failed')


## 3 — Add repo to path and import TiTok

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

REPO = '1d-tokenizer'
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from modeling.titok import TiTok   # correct import path
print('TiTok imported: OK')
print(f'Module: {TiTok.__module__}')


## 4 — Load model weights

Tries in order:
1. `yucornetto/tokenizer_titok_b64_imagenet` via `from_pretrained` (safetensors + config.json)
2. Other `yucornetto` variants (L-32, S-128)
3. `.bin` weight files from `fun-research/TiTok` + local YAML config (reliable fallback)


In [ ]:
import torch
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

DEVICE = 'cuda'
model = None; N_TOKENS = None; CODEBOOK_SIZE = None; MODEL_ID = None


def _test_model(m):
    """Verify model works with a dummy forward pass and return (N_TOKENS, codebook_size)."""
    m = m.to(DEVICE).eval()
    with torch.no_grad():
        zq, rd = m.encode(torch.zeros(1, 3, 256, 256, device=DEVICE))
    n  = zq.shape[-1]                          # num_latent_tokens
    cb = m.quantize.embedding.num_embeddings   # codebook size
    return m, n, cb


def try_from_pretrained(hf_id):
    """Use PyTorchModelHubMixin.from_pretrained — downloads config.json + model.safetensors."""
    global model, N_TOKENS, CODEBOOK_SIZE, MODEL_ID
    print(f'  {hf_id} ...', end=' ')
    try:
        m, n, cb = _test_model(TiTok.from_pretrained(hf_id))
        model, N_TOKENS, CODEBOOK_SIZE, MODEL_ID = m, n, cb, hf_id
        print(f'OK  N={n}  Q={cb}')
        return True
    except Exception as e:
        print(f'failed: {e}')
        return False


def try_bin_download(repo_id, filename, cfg_yaml):
    """Download .bin weights from fun-research/TiTok and load with local YAML config."""
    global model, N_TOKENS, CODEBOOK_SIZE, MODEL_ID
    print(f'  fun-research/TiTok/{filename} ...', end=' ')
    try:
        ckpt = hf_hub_download(repo_id=repo_id, filename=filename, local_dir='/tmp/titok')
        cfg  = OmegaConf.load(cfg_yaml)
        m    = TiTok(cfg)
        sd   = torch.load(ckpt, map_location='cpu', weights_only=True)
        m.load_state_dict(sd, strict=False)
        m, n, cb = _test_model(m)
        model, N_TOKENS, CODEBOOK_SIZE, MODEL_ID = m, n, cb, filename.replace('.bin', '')
        print(f'OK  N={n}  Q={cb}')
        return True
    except Exception as e:
        print(f'failed: {e}')
        return False


print('Loading tokenizer model...')

# ── Option 1: from_pretrained (preferred) ────────────────────────────────────
for hf_id in [
    'yucornetto/tokenizer_titok_b64_imagenet',
    'yucornetto/tokenizer_titok_l32_imagenet',
    'yucornetto/tokenizer_titok_s128_imagenet',
]:
    if try_from_pretrained(hf_id):
        break

# ── Option 2: .bin from fun-research/TiTok (reliable public fallback) ─────────
if model is None:
    CFG = '1d-tokenizer/configs/infer/TiTok'
    for fname, yaml in [
        ('tokenizer_titok_b64.bin', f'{CFG}/titok_b64.yaml'),
        ('tokenizer_titok_l32.bin', f'{CFG}/titok_l32.yaml'),
        ('tokenizer_titok_s128.bin', f'{CFG}/titok_s128.yaml'),
    ]:
        if try_bin_download('fun-research/TiTok', fname, yaml):
            break

if model is None:
    raise RuntimeError(
        'All model candidates failed.\n'
        'Check Colab has internet access and HuggingFace is reachable.\n'
        'Manual fallback: upload tokenizer_titok_b64.bin from '
        'https://huggingface.co/fun-research/TiTok and re-run.'
    )

BITS_PER_TOKEN = max(1, (CODEBOOK_SIZE - 1).bit_length())
print(f'\nReady: {MODEL_ID}  |  N={N_TOKENS} tokens  |  Q={CODEBOOK_SIZE}  |  {BITS_PER_TOKEN} bits/token')


## 5 — Tokenize & prefix-decode (core TCLA functions)

Key facts about TiTok's API:
- Input range: **[0, 1]** (divide by 255, no mean/std normalisation)
- Output range: **[0, 1]** (clamp then multiply by 255)
- Prefix decode: keep `z_quantized` shape `(1, C, 1, N)` — fill tail slots with the **mean codebook embedding** (neutral) and call `model.decode()`


In [ ]:
import numpy as np
from PIL import Image
import torch, warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

IMG_SIZE = 256


def _to_tensor(pil_img):
    """PIL image -> (1,3,256,256) float32 in [0,1].  TiTok expects [0,1], not [-1,1]."""
    arr = np.array(pil_img.convert('RGB').resize((IMG_SIZE, IMG_SIZE))).astype(np.float32)
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).div(255.0).to(DEVICE)


def _to_uint8(tensor):
    """(1,3,H,W) float32 [0,1] -> (H,W,3) uint8."""
    arr = torch.clamp(tensor, 0.0, 1.0).squeeze(0).permute(1, 2, 0).cpu().float().numpy()
    return (arr * 255.0).astype(np.uint8)


# Mean codebook embedding — used as neutral fill for masked tail positions
with torch.no_grad():
    _MEAN_EMB = model.quantize.embedding.weight.mean(dim=0).to(DEVICE)  # shape (token_size,)
print(f'Mean codebook embedding: {_MEAN_EMB.shape}  device={_MEAN_EMB.device}')


def tokenize(pil_img):
    """
    Encode a PIL image.
    Returns:
        zq  : (1, C, 1, N)  float32  continuous quantized latents
        ids : (N,)           int64    discrete token indices
    """
    x = _to_tensor(pil_img)
    with torch.no_grad():
        zq, rd = model.encode(x)
    ids = rd['min_encoding_indices'].reshape(-1).cpu().numpy().astype(np.int64)
    return zq, ids


def decode_prefix(zq, k):
    """
    Reconstruct image from first k tokens.

    The decoder expects the full (1, C, 1, N) tensor — we keep the shape but
    replace tail positions [k..N] with the mean codebook embedding, which acts
    as a learned 'no information here' signal.

    With One-D-Piece (TTD-trained) the head tokens are semantically ordered
    so even small k gives a recognisable image.  With base TiTok the ordering
    is not guaranteed but degradation is still graceful.

    Returns (H, W, 3) uint8 numpy array.
    """
    k = max(1, min(k, N_TOKENS))
    z = zq.clone()                             # (1, C, 1, N)
    if k < N_TOKENS:
        # z[:, :, 0, k:] has shape (1, C, N-k)
        z[:, :, 0, k:] = _MEAN_EMB.view(1, -1, 1).expand(1, -1, N_TOKENS - k)
    with torch.no_grad():
        recon = model.decode(z)                # (1, 3, H, W) in [0, 1]
    return _to_uint8(recon)


# ── Visual sanity check ─────────────────────────────────────────────────────
# Try several public image URLs; fall back to a generated image if all fail
import urllib.request
TEST_URLS = [
    'https://farm3.staticflickr.com/2879/9423oswaldo.jpg',          # will 404 -> skip
    'https://picsum.photos/seed/tcla/320/320.jpg',                   # Lorem Picsum (reliable)
    'https://images.unsplash.com/photo-1533738363-b7f9aef128ce?w=320',
]

def _synthetic_test_img():
    """Generate a colourful synthetic image so the sanity check always works."""
    arr = np.zeros((256, 256, 3), dtype=np.uint8)
    for y in range(256):
        for x in range(256):
            arr[y, x] = [
                int(128 + 127 * np.sin(2*np.pi*y/256)),
                int(128 + 127 * np.sin(2*np.pi*x/256)),
                int(128 + 127 * np.sin(2*np.pi*(x+y)/256)),
            ]
    from PIL import ImageDraw
    img = Image.fromarray(arr)
    d   = ImageDraw.Draw(img)
    d.ellipse([80, 80, 176, 176], fill=(255, 220, 50))
    d.rectangle([100, 150, 156, 200], fill=(200, 80, 80))
    return img

test_img = None
for url in TEST_URLS:
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=8) as r, open('/tmp/test.jpg', 'wb') as f:
            f.write(r.read())
        test_img = Image.open('/tmp/test.jpg').convert('RGB')
        print(f'Test image downloaded from: {url}')
        break
    except Exception as e:
        print(f'URL failed ({e}), trying next...')

if test_img is None:
    test_img = _synthetic_test_img()
    print('Using synthetic test image (all URLs failed)')

print('Testing prefix reconstruction...')
zq_t, ids_t = tokenize(test_img)
print(f'  z_q shape : {zq_t.shape}')
print(f'  token ids : shape={ids_t.shape}  range=[{ids_t.min()}, {ids_t.max()}]')

k_vals = sorted(set([1, 4, 8, 16, 32, N_TOKENS // 2, N_TOKENS]))
recons = [decode_prefix(zq_t, k) for k in k_vals]

fig, axes = plt.subplots(1, len(k_vals) + 1, figsize=(3.2 * (len(k_vals) + 1), 3.5))
fig.patch.set_facecolor('#0d0d0d')
axes[0].imshow(np.array(test_img.resize((256, 256))))
axes[0].set_title('Original', color='white', fontsize=9)
axes[0].axis('off')
for ax, k, rc in zip(axes[1:], k_vals, recons):
    ax.imshow(rc)
    ax.set_title(f'k={k}\n{k * 100 // N_TOKENS}%', color='#69f0ae', fontsize=9)
    ax.axis('off')
plt.suptitle(f'Prefix reconstruction — {MODEL_ID}', color='white')
plt.tight_layout()
plt.savefig('/tmp/prefix_test.png', dpi=110, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Prefix reconstruction: OK')


## 6 — Load video (Big Buck Bunny or synthetic fallback)

In [ ]:
import subprocess, os

VIDEO_SRC  = '/tmp/src.mp4'
DURATION_S = 10
FPS        = 25
N_FRAMES   = DURATION_S * FPS


def download_bbunny():
    print('Downloading Big Buck Bunny (CC BY 3.0)...')
    r = subprocess.run([
        'ffmpeg', '-y', '-i',
        'https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/BigBuckBunny.mp4',
        '-ss', '62', '-t', str(DURATION_S),
        '-vf', f'scale={IMG_SIZE}:{IMG_SIZE}:force_original_aspect_ratio=disable',
        '-r', str(FPS), '-an', '-c:v', 'libx264', VIDEO_SRC
    ], capture_output=True, timeout=300)
    return r.returncode == 0 and os.path.exists(VIDEO_SRC) and os.path.getsize(VIDEO_SRC) > 10_000


def make_synthetic():
    from PIL import ImageDraw
    import imageio
    print('Generating synthetic video...')
    w = imageio.get_writer(VIDEO_SRC, fps=FPS, codec='libx264', quality=8, macro_block_size=1)
    for i in range(N_FRAMES):
        t   = i / FPS
        arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        for y in range(IMG_SIZE):
            h = (y / IMG_SIZE + t * 0.07) % 1.0
            arr[y] = [int(128 + 127 * np.sin(2 * np.pi * (h + j / 3))) % 256 for j in range(3)]
        pil = Image.fromarray(arr)
        drw = ImageDraw.Draw(pil)
        bx  = int(128 + 90 * np.sin(t * 2.0))
        by  = int(100 + 60 * abs(np.sin(t * 2.8)))
        drw.ellipse([bx-22, by-22, bx+22, by+22], fill=(255, 220, 30))
        drw.rectangle([50, 155, 206, 200], fill=(0, 0, 0))
        drw.text((128, 177),
                 f'{int(t)//60:02d}:{int(t)%60:02d}.{int((t%1)*10)}',
                 fill=(0, 255, 100), anchor='mm')
        w.append_data(np.array(pil))
    w.close()


if download_bbunny():
    print(f'Video downloaded: {os.path.getsize(VIDEO_SRC) // 1024} KB')
else:
    make_synthetic()

# Extract raw frames
raw = subprocess.run([
    'ffmpeg', '-y', '-i', VIDEO_SRC,
    '-vf', f'scale={IMG_SIZE}:{IMG_SIZE},fps={FPS}',
    '-t', str(DURATION_S),
    '-f', 'rawvideo', '-pix_fmt', 'rgb24', 'pipe:1'
], capture_output=True, timeout=60).stdout

n_avail = len(raw) // (IMG_SIZE * IMG_SIZE * 3)
frames  = [
    np.frombuffer(raw[i * IMG_SIZE*IMG_SIZE*3 : (i+1) * IMG_SIZE*IMG_SIZE*3], np.uint8)
      .reshape(IMG_SIZE, IMG_SIZE, 3).copy()
    for i in range(min(n_avail, N_FRAMES))
]
N_FRAMES = len(frames)
print(f'{N_FRAMES} frames loaded  ({N_FRAMES / FPS:.1f}s @ {FPS} fps  {IMG_SIZE}x{IMG_SIZE})')

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.patch.set_facecolor('#0d0d0d')
for ax, idx in zip(axes, [0, N_FRAMES//5, 2*N_FRAMES//5, 3*N_FRAMES//5, N_FRAMES-1]):
    ax.imshow(frames[idx])
    ax.set_title(f't={idx/FPS:.1f}s', color='white', fontsize=9)
    ax.axis('off')
plt.suptitle('Source video frames', color='white')
plt.tight_layout(); plt.show()


## 7 — WiFi channel model (802.11ax MCS table)

In [ ]:
from dataclasses import dataclass

# 802.11ax: (sensitivity_snr_dB, phy_rate_Mbps, label)
MCS_TABLE = [
    ( 7.0,   7.3, 'MCS0 BPSK 1/2'),
    (10.0,  14.4, 'MCS1 QPSK 1/2'),
    (13.0,  21.7, 'MCS2 QPSK 3/4'),
    (16.0,  28.9, 'MCS3 16QAM 1/2'),
    (19.0,  43.3, 'MCS4 16QAM 3/4'),
    (22.0,  57.8, 'MCS5 64QAM 2/3'),
    (24.0,  65.0, 'MCS6 64QAM 3/4'),
    (26.0,  72.2, 'MCS7 64QAM 5/6'),
    (29.0,  86.7, 'MCS8 256QAM 3/4'),
    (34.0, 120.0, 'MCS9 1024QAM 5/6'),
]
MAC_EFF = 0.75  # protocol overhead factor


@dataclass
class ChState:
    idx: int; time_s: float; snr_db: float
    rate_mbps: float; mcs_label: str
    per: float; avail_bits: float; is_lost: bool


def snr_to_mcs(snr):
    best = MCS_TABLE[0]
    for row in MCS_TABLE:
        if snr >= row[0]: best = row
    return best[1], best[2], best[0]  # rate, label, threshold


def build_channel(n_frames, fps, seed=42):
    """Interference scenario: good -> sudden drop -> sustained low -> recovery."""
    t   = np.linspace(0, 1, n_frames)
    snr = np.zeros(n_frames)
    for i, ti in enumerate(t):
        if   ti < 0.25:  snr[i] = 30.0
        elif ti < 0.30:  snr[i] = 30.0 - 28.0 * ((ti - 0.25) / 0.05)   # sharp drop
        elif ti < 0.60:  snr[i] = 2.0  +  4.0 * abs(np.sin(2*np.pi*ti*3))  # jittery low
        elif ti < 0.70:  snr[i] = 6.0  + 24.0 * ((ti - 0.60) / 0.10)   # recovery
        else:            snr[i] = 30.0
    rng  = np.random.default_rng(seed)
    snr += rng.standard_normal(n_frames) * 0.6
    fp   = 1.0 / fps
    states = []
    for i, s in enumerate(snr):
        rate, label, thr = snr_to_mcs(float(s))
        per   = float(1.0 / (1.0 + np.exp(2.5 * (s - thr))))
        avail = rate * 1e6 * fp * MAC_EFF * (1.0 - per)
        states.append(ChState(i, i/fps, float(s), rate, label, per, avail,
                               bool(rng.random() < per)))
    return states


channel = build_channel(N_FRAMES, FPS)
snrs    = [c.snr_db for c in channel]
print(f'SNR range : {min(snrs):.1f} – {max(snrs):.1f} dB')
print(f'Frames <12 dB: {sum(s < 12 for s in snrs)} / {N_FRAMES}'
      f'  ({sum(s<12 for s in snrs)/N_FRAMES*100:.0f}%)')

fig, ax = plt.subplots(figsize=(12, 2.5))
fig.patch.set_facecolor('#0d0d0d'); ax.set_facecolor('#141414')
ax.plot([c.time_s for c in channel], snrs, '#4fc3f7', lw=1.5)
ax.fill_between([c.time_s for c in channel], snrs, alpha=0.12, color='#4fc3f7')
ax.axhline(12, color='#ff7043', ls='--', lw=1.2, label='H.265 minimum (~12 dB)')
ax.set_ylabel('SNR (dB)', color='white'); ax.set_xlabel('Time (s)', color='white')
ax.tick_params(colors='white'); ax.spines[:].set_color('#333')
ax.legend(facecolor='#1e1e1e', labelcolor='white')
ax.set_title('WiFi channel — interference event at t = 2.5 – 6 s', color='white')
plt.tight_layout(); plt.show()


## 8 — H.265 simulation

Estimates per-frame H.265 size (JPEG proxy), checks if it fits in the channel budget.
A dropped I-frame cascades: all subsequent P-frames until the next I-frame are also lost.

In [ ]:
import io as _io, time as _t
from dataclasses import dataclass, field as fld

GOP_SIZE     = 25     # I-frame every 25 frames = 1 s at 25 fps
FREEZE_DECAY = 0.055  # QoE drops this much per frozen frame


@dataclass
class H265R:
    idx: int; frozen: bool; freeze_dur: int
    enc_bytes: int; frame_type: str; qoe: float; snr_db: float
    display: np.ndarray = fld(default=None, repr=False)


def h265_enc_size(frame_rgb, is_keyframe):
    """Estimate H.265 frame size via JPEG proxy (fast, no ffmpeg subprocess)."""
    buf = _io.BytesIO()
    Image.fromarray(frame_rgb).save(buf, 'JPEG', quality=42)
    jpeg = buf.tell()
    return int(jpeg * (0.55 if is_keyframe else 0.08))


print(f'Simulating H.265 ({N_FRAMES} frames)...')
t0 = _t.time()
h265_results = []
last_good    = frames[0].copy()
freeze_dur   = 0
i_cascade    = False   # True while we're waiting for the next I-frame after a dropped I

for i, (frm, ch) in enumerate(zip(frames, channel)):
    is_kf  = (i % GOP_SIZE == 0)
    ftype  = 'I' if is_kf else 'P'
    enc_b  = h265_enc_size(frm, is_kf)
    fits   = (enc_b * 8) <= ch.avail_bits
    ok     = fits and not ch.is_lost
    if i_cascade and not is_kf:
        ok = False                           # cascade: P-frames lost until next I
    if ok:
        last_good, freeze_dur, i_cascade = frm.copy(), 0, False
        qoe = 1.0
    else:
        freeze_dur += 1
        qoe = max(0.0, 1.0 - freeze_dur * FREEZE_DECAY)
        if is_kf: i_cascade = True           # dropped I-frame starts cascade
    h265_results.append(H265R(i, not ok, freeze_dur, enc_b, ftype, qoe,
                               ch.snr_db, last_good.copy()))

n_frozen = sum(r.frozen for r in h265_results)
print(f'Done in {_t.time()-t0:.1f}s')
print(f'Frozen frames : {n_frozen}/{N_FRAMES} ({n_frozen/N_FRAMES*100:.0f}%) = {n_frozen/FPS:.1f}s')
print(f'Avg QoE       : {np.mean([r.qoe for r in h265_results]):.3f}')


## 9 — TCLA simulation (real tokenizer + prefix decode on GPU)

For each frame: compute `k_max` from the channel budget, tokenize the frame, reconstruct from the first `k_max` tokens.  `k` is never zero — the decoder always produces a valid image.

In [ ]:
from dataclasses import dataclass, field as fld

K_MIN        = max(1, N_TOKENS // 32)  # floor: e.g. 2 for N=64, 8 for N=256
QOE_FLOOR    = 0.15
QOE_ALPHA    = 3.5


def qoe_of_k(k):
    """Analytical QoE(k) curve — concave, monotone, floor > 0."""
    return QOE_FLOOR + (1 - QOE_FLOOR) * (1 - np.exp(-QOE_ALPHA * min(k / N_TOKENS, 1.0)))


def k_from_channel(ch):
    """Max tokens deliverable within the frame budget."""
    return max(K_MIN, min(N_TOKENS, int(ch.avail_bits / BITS_PER_TOKEN)))


@dataclass
class TCLAR:
    idx: int; k: int; N: int; qoe: float; snr_db: float
    enc_ms: float; dec_ms: float
    display: np.ndarray = fld(default=None, repr=False)


print(f'Simulating TCLA with {MODEL_ID}  ({N_FRAMES} frames)...')
print(f'K_MIN={K_MIN}  N_TOKENS={N_TOKENS}  BITS_PER_TOKEN={BITS_PER_TOKEN}')
t0 = _t.time()
tcla_results = []

for i, (frm, ch) in enumerate(zip(frames, channel)):
    k   = k_from_channel(ch)
    pil = Image.fromarray(frm)

    te = _t.time()
    zq, ids = tokenize(pil)
    enc_ms  = (_t.time() - te) * 1000

    td = _t.time()
    recon  = decode_prefix(zq, k)
    dec_ms = (_t.time() - td) * 1000

    tcla_results.append(TCLAR(i, k, N_TOKENS, qoe_of_k(k), ch.snr_db,
                               enc_ms, dec_ms, recon))

    if i % 25 == 0:
        elapsed = _t.time() - t0
        eta     = elapsed / (i + 1) * (N_FRAMES - i - 1)
        print(f'  [{i:3d}/{N_FRAMES}]  k={k:3d}/{N_TOKENS} ({k*100//N_TOKENS:3d}%)'
              f'  QoE={qoe_of_k(k):.2f}  SNR={ch.snr_db:+.0f}dB'
              f'  enc={enc_ms:.0f}ms dec={dec_ms:.0f}ms  ETA={eta:.0f}s')

total = _t.time() - t0
print(f'\nDone in {total:.1f}s  ({total/N_FRAMES*1000:.0f} ms/frame)')
print(f'Min QoE : {min(r.qoe for r in tcla_results):.3f}  (never zero)')
print(f'Min k   : {min(r.k for r in tcla_results)}/{N_TOKENS}')
print(f'TCLA never freezes: OK')


## 10 — Frame comparison at key moments

In [ ]:
# Sample moments: before, onset, deep interference, mid-recovery, recovered
key_times = [1.0, 2.8, 4.0, 5.5, 7.5, 9.5]
key_idx   = [min(N_FRAMES - 1, int(t * FPS)) for t in key_times]
n_cols    = len(key_idx)

fig = plt.figure(figsize=(3.2 * n_cols, 10))
fig.patch.set_facecolor('#0a0a0a')
gs  = gridspec.GridSpec(3, n_cols, hspace=0.05, wspace=0.03)

for col, idx in enumerate(key_idx):
    ch  = channel[idx]
    hr  = h265_results[idx]
    tr  = tcla_results[idx]
    imgs = [frames[idx], hr.display, tr.display]
    row_colors  = ['#ffffff', '#ff5252', '#40c4ff']
    row_labels  = ['Original', 'H.265', 'TCLA']

    for row, (img, lc, ln) in enumerate(zip(imgs, row_colors, row_labels)):
        ax = fig.add_subplot(gs[row, col])
        ax.imshow(img); ax.axis('off')

        if row == 0:
            ax.set_title(f't={ch.time_s:.1f}s\n{ch.snr_db:+.0f} dB',
                         color='white', fontsize=8.5, pad=3)
        if col == 0:
            ax.set_ylabel(ln, color=lc, fontsize=9.5,
                          rotation=0, labelpad=44, va='center')

        if row == 1:
            txt = (f'FROZEN  {hr.freeze_dur/FPS:.1f}s' if hr.frozen
                   else f'OK  QoE={hr.qoe:.2f}')
            bg  = '#cc2000' if hr.frozen else '#006630'
            ax.text(0.5, 0.04, txt, transform=ax.transAxes,
                    ha='center', va='bottom', color='white', fontsize=7.5,
                    bbox=dict(boxstyle='round,pad=0.25', fc=bg, alpha=0.9))
        elif row == 2:
            ax.text(0.5, 0.04, f'k={tr.k}/{N_TOKENS}  QoE={tr.qoe:.2f}',
                    transform=ax.transAxes,
                    ha='center', va='bottom', color='white', fontsize=7.5,
                    bbox=dict(boxstyle='round,pad=0.25', fc='#003488', alpha=0.9))

plt.suptitle(f'Original  |  H.265  |  TokCom TCLA  —  {MODEL_ID}',
             color='white', fontsize=11, y=0.99)
plt.savefig('/tmp/frame_comparison.png', dpi=140,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()


## 11 — QoE over time: the cliff vs slope comparison

In [ ]:
times  = [c.time_s for c in channel]
snrs   = [c.snr_db for c in channel]
h_qoe  = [r.qoe    for r in h265_results]
t_qoe  = [r.qoe    for r in tcla_results]
k_frac = [r.k / r.N for r in tcla_results]

# Find freeze spans for shading
spans, in_f, s0 = [], False, 0.0
for r in h265_results:
    if r.frozen and not in_f:      s0, in_f = r.idx / FPS, True
    elif not r.frozen and in_f:    spans.append((s0, r.idx / FPS)); in_f = False
if in_f: spans.append((s0, times[-1]))

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.patch.set_facecolor('#0a0a0a')
for ax in axes:
    ax.set_facecolor('#141414'); ax.tick_params(colors='#bbb')
    ax.spines[:].set_color('#2a2a2a')
    for lbl in ax.get_xticklabels() + ax.get_yticklabels(): lbl.set_color('#bbb')

# ── SNR ──────────────────────────────────────────────────────────────────────
axes[0].plot(times, snrs, '#4fc3f7', lw=1.8)
axes[0].fill_between(times, snrs, alpha=0.12, color='#4fc3f7')
axes[0].axhline(12, color='#ff7043', ls='--', lw=1.2, label='H.265 minimum SNR')
for s, e in spans: axes[0].axvspan(s, e, alpha=0.14, color='#ff4444')
axes[0].set_ylabel('SNR (dB)', color='#bbb'); axes[0].set_ylim(-2, 38)
axes[0].legend(facecolor='#1e1e1e', labelcolor='#bbb', fontsize=9)
axes[0].set_title('WiFi channel SNR  (shaded = H.265 freeze region)',
                   color='white', fontsize=10)

# ── QoE ──────────────────────────────────────────────────────────────────────
axes[1].fill_between(times, h_qoe, alpha=0.22, color='#ff5252')
axes[1].plot(times, h_qoe, '#ff5252', lw=2.2,
             label=f'H.265   avg QoE = {np.mean(h_qoe):.2f}')
axes[1].fill_between(times, t_qoe, alpha=0.18, color='#69f0ae')
axes[1].plot(times, t_qoe, '#69f0ae', lw=2.2,
             label=f'TCLA    avg QoE = {np.mean(t_qoe):.2f}')
for s, e in spans:
    axes[1].axvspan(s, e, alpha=0.08, color='#ff4444')
    axes[1].annotate('FREEZE', xy=((s + e) / 2, 0.07),
                     ha='center', color='#ff8a80', fontsize=8)
axes[1].set_ylabel('QoE', color='#bbb'); axes[1].set_ylim(-0.05, 1.12)
axes[1].legend(facecolor='#1e1e1e', labelcolor='#bbb', fontsize=10)
axes[1].set_title('QoE  —  H.265 cliff  vs  TCLA graceful slope',
                   color='white', fontsize=10)

# ── k(t) ─────────────────────────────────────────────────────────────────────
axes[2].fill_between(times, k_frac, alpha=0.28, color='#b39ddb')
axes[2].plot(times, k_frac, '#b39ddb', lw=1.8, label='k / N  (token fraction)')
axes[2].axhline(1.0, color='white', lw=0.5, ls=':', alpha=0.3)
for s, e in spans: axes[2].axvspan(s, e, alpha=0.08, color='#ff4444')
axes[2].set_ylabel('k / N', color='#bbb'); axes[2].set_xlabel('Time (s)', color='#bbb')
axes[2].set_ylim(-0.05, 1.15)
axes[2].set_yticks([0, .25, .5, .75, 1.0])
axes[2].set_yticklabels(['0%', '25%', '50%', '75%', '100%'], color='#bbb')
axes[2].legend(facecolor='#1e1e1e', labelcolor='#bbb', fontsize=9)
axes[2].set_title('TCLA token count k(t) — adapts every frame, never zero',
                   color='white', fontsize=10)

hn = sum(r.frozen for r in h265_results)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.suptitle(
    f'H.265 vs TokCom TCLA  |  model: {MODEL_ID}  |  N = {N_TOKENS} tokens\n'
    f'H.265: {hn} frozen frames ({hn/FPS:.1f}s)  |  TCLA: 0 frozen  |  '
    f'QoE gain: +{(np.mean(t_qoe) - np.mean(h_qoe))*100:.0f}%',
    color='white', fontsize=11, fontweight='bold'
)
plt.savefig('/tmp/qoe_plot.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

print(f'\n{"="*55}')
print(f'  H.265  frozen : {hn}/{N_FRAMES} ({hn/N_FRAMES*100:.0f}%) = {hn/FPS:.1f}s')
print(f'  H.265  avg QoE: {np.mean(h_qoe):.3f}')
print(f'  TCLA   frozen : 0  (never)')
print(f'  TCLA   avg QoE: {np.mean(t_qoe):.3f}')
print(f'  TCLA   min k  : {min(r.k for r in tcla_results)}/{N_TOKENS}')
print(f'  QoE gain      : +{(np.mean(t_qoe)-np.mean(h_qoe))*100:.0f}%')
print(f'{"="*55}')


## 12 — Render side-by-side comparison MP4

In [ ]:
from PIL import ImageDraw, ImageFont

HUD_H = 170
OUT_W = IMG_SIZE * 2
OUT_H = IMG_SIZE + HUD_H
OUT_VID = '/tmp/tcla_vs_h265_REAL.mp4'

try:
    FONT_MED = ImageFont.truetype(
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 13)
    FONT_SM  = ImageFont.truetype(
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 10)
except Exception:
    FONT_MED = FONT_SM = ImageFont.load_default()


def draw_bar(D, x, y, w, h, frac, fg, bg=(35, 35, 35)):
    frac = max(0.0, min(1.0, frac))
    D.rectangle([x, y, x + w, y + h], fill=bg)
    D.rectangle([x, y, x + int(w * frac), y + h], fill=fg)


def compose_frame(i, hr, tr, ch):
    C = Image.new('RGB', (OUT_W, OUT_H), (10, 10, 10))
    D = ImageDraw.Draw(C)

    # ── H.265 panel (left) ────────────────────────────────────────────────────
    hi = Image.fromarray(hr.display)
    if hr.frozen:
        ov = Image.new('RGBA', hi.size, (20, 50, 180, 100))
        hi = Image.alpha_composite(hi.convert('RGBA'), ov).convert('RGB')
        d2 = ImageDraw.Draw(hi)
        d2.text((IMG_SIZE // 2, IMG_SIZE // 2 - 8), 'FROZEN',
                font=FONT_MED, fill=(160, 200, 255), anchor='mm')
        d2.text((IMG_SIZE // 2, IMG_SIZE // 2 + 12),
                f'{hr.freeze_dur / FPS:.1f}s',
                font=FONT_MED, fill=(160, 200, 255), anchor='mm')
    C.paste(hi, (0, 0))

    # ── TCLA panel (right) ────────────────────────────────────────────────────
    C.paste(Image.fromarray(tr.display), (IMG_SIZE, 0))
    D.line([(IMG_SIZE, 0), (IMG_SIZE, IMG_SIZE)], fill=(45, 45, 45), width=2)

    # Label bars
    D.rectangle([0, 0, IMG_SIZE, 22], fill=(12, 12, 12))
    D.text((IMG_SIZE // 2, 11), 'H.265  (conventional)',
           font=FONT_SM, fill=(255, 110, 80), anchor='mm')
    D.rectangle([IMG_SIZE, 0, OUT_W, 22], fill=(12, 12, 12))
    D.text((IMG_SIZE + IMG_SIZE // 2, 11),
           f'TokCom TCLA   k = {tr.k} / {N_TOKENS}',
           font=FONT_SM, fill=(70, 190, 255), anchor='mm')

    # ── HUD ──────────────────────────────────────────────────────────────────
    y0 = IMG_SIZE
    D.rectangle([0, y0, OUT_W, OUT_H], fill=(14, 14, 14))
    D.line([(0, y0), (OUT_W, y0)], fill=(55, 55, 55))

    # SNR bar
    sf = max(0.0, min(1.0, ch.snr_db / 35.0))
    D.text((10, y0 + 18), 'WiFi SNR', font=FONT_SM, fill=(140, 140, 140))
    draw_bar(D, 90, y0 + 11, 360, 13, sf,
             fg=(int(255 * (1 - sf)), int(200 * sf), 50))
    D.text((458, y0 + 18),
           f'{ch.snr_db:.1f} dB   {ch.mcs_label}',
           font=FONT_SM, fill=(180, 180, 180))

    # H.265 status row
    hc  = (255, 80, 60) if hr.frozen else (70, 210, 80)
    htx = (f'FROZEN   {hr.freeze_dur / FPS:.1f}s   QoE={hr.qoe:.2f}'
           if hr.frozen else f'OK  [{hr.frame_type}-frame]   QoE={hr.qoe:.2f}')
    D.text((10,  y0 + 52), 'H.265', font=FONT_MED, fill=(255, 90, 70))
    D.text((78,  y0 + 52), htx, font=FONT_MED, fill=hc)
    draw_bar(D, 460, y0 + 46, 148, 10, hr.qoe,
             fg=(int(255 * (1 - hr.qoe)), int(210 * hr.qoe), 50))

    # TCLA status row
    D.text((10, y0 + 86), 'TCLA', font=FONT_MED, fill=(70, 180, 255))
    D.text((78, y0 + 86),
           f'k={tr.k}/{N_TOKENS} ({tr.k * 100 // N_TOKENS}%)   QoE={tr.qoe:.2f}',
           font=FONT_MED, fill=(70, 180, 255))
    draw_bar(D, 460, y0 + 80, 148, 10, tr.k / N_TOKENS,
             fg=(50, int(80 + 175 * (tr.k / N_TOKENS)), 255))

    # Footer
    D.text((10, y0 + 120),
           f't = {ch.time_s:.2f}s   frame #{i}   |   '
           f'TCLA: quality adapts, never freezes',
           font=FONT_SM, fill=(80, 80, 80))

    return np.array(C)


print(f'Rendering {N_FRAMES} frames -> {OUT_VID}')
t0 = _t.time()
proc = subprocess.Popen([
    'ffmpeg', '-y',
    '-f', 'rawvideo', '-pix_fmt', 'rgb24',
    '-s', f'{OUT_W}x{OUT_H}', '-r', str(FPS),
    '-i', 'pipe:0',
    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p',
    '-crf', '17', '-preset', 'fast',
    OUT_VID
], stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)

for i, (hr, tr, ch) in enumerate(zip(h265_results, tcla_results, channel)):
    proc.stdin.write(compose_frame(i, hr, tr, ch).tobytes())
    if i % 50 == 0:
        print(f'  {i}/{N_FRAMES}...', end='\r')

proc.stdin.close()
proc.wait()
size_mb = os.path.getsize(OUT_VID) / 1e6
print(f'\nDone in {_t.time()-t0:.0f}s  ->  {OUT_VID}  ({size_mb:.1f} MB)')


## 13 — Play inline & download results

In [ ]:
from IPython.display import HTML, display, Image as IPImg
import base64

with open(OUT_VID, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<video width='900' controls autoplay loop style='background:#000; border-radius:8px;'>
  <source src='data:video/mp4;base64,{b64}' type='video/mp4'>
</video>
<p style='color:#999; font-size:12px; font-family:monospace; margin-top:6px;'>
  Left: H.265 (freezes when channel is weak) &nbsp;|&nbsp;
  Right: TokCom TCLA (always moving — quality adapts with k)<br>
  Worst case: k={min(r.k for r in tcla_results)}/{N_TOKENS} tokens sent — decoder fills the rest.
</p>
"""))


In [ ]:
display(IPImg('/tmp/qoe_plot.png', width=900))
display(IPImg('/tmp/frame_comparison.png', width=900))
display(IPImg('/tmp/prefix_test.png', width=700))


In [ ]:
from google.colab import files
print('Downloading outputs...')
files.download(OUT_VID)
files.download('/tmp/qoe_plot.png')
files.download('/tmp/frame_comparison.png')
print('Done!')


## Summary

| | H.265 | TokCom TCLA |
|---|---|---|
| **When channel drops** | Frame dropped → hold last → ❄ FREEZE | k reduced → blurry but moving |
| **Adaptation** | None — fixed bitrate target | k recalculated every frame |
| **Recovery** | Waits for next I-frame | Instant — k increases next frame |
| **Minimum quality** | Zero (frozen) | QoE floor ≈ 0.15 (decoder generates from few tokens) |
| **Payload at worst SNR** | Would need full frame (~50 KB I-frame) | k × 1.5 B = as low as ~3–12 bytes |

**Why TCLA works:** TiTok's decoder is a cross-attention transformer. The `N` latent tokens are keys/values; the output image patches are queries. Pass fewer tokens → queries attend to less information → blurrier but coherent reconstruction. With One-D-Piece (Tail Token Drop training) the head tokens are semantically ordered, so even `k=8` gives a recognisable image — that is the MLLM reconstruction floor in action.

**Next step:** replace the simulated channel with a real WiFi link using the `transmitter.py` / `receiver.py` scripts from the demo package.